In [ ]:
!pip install pydicom
!pip install wget
!pip install open_clip_torch
!pip install --upgrade transformers

In [ ]:
import torch.nn as nn
class Denoiser(nn.Module):
    def __init__(self, channels=3, num_of_layers=17):
        super().__init__()
        kernel_size = 3
        padding = 1
        features = 64

        layers = [
            nn.Conv2d(channels, features, kernel_size, padding=padding, bias=False),
            nn.ReLU(inplace=True)
        ]

        for _ in range(num_of_layers - 2):
            layers += [
                nn.Conv2d(features, features, kernel_size, padding=padding, bias=False),
                nn.BatchNorm2d(features),
                nn.ReLU(inplace=True)
            ]

        layers.append(
            nn.Conv2d(features, channels, kernel_size, padding=padding, bias=False)
        )

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)  

In [ ]:
from modules.dataset.factory import DatasetFactory
from modules.utils.constants import MODEL_TRANSFORMS, DEFAULT_TEMPLATES, RSNA_CLASS_PROMPTS, RSNA_CLASS_PROMPTS, SIZE_TRANSFORM, TENSOR_NORMALIZE_TRANSFORM,DATA_ROOT, ENTREP_CLASS_PROMPTS
from modules.models.factory import ModelFactory
from modules.utils.helpers import _extract_label, load_open_clip_model

In [ ]:
from torch.utils.data import Dataset
from torchvision import transforms
import csv
import os
from PIL import Image
_toTensor = transforms.ToTensor()
from torch.utils.data import Dataset

class MiMic(Dataset):
    def __init__(self, size_transform):
        from datasets import load_dataset
        self.ds = load_dataset(
            "itsanmolgupta/mimic-cxr-dataset",
            split="train",
            streaming=False
        )

        self.size_transform = size_transform
        self.samples = []

        for i in range(len(self.ds)):
            if self.ds[i]["findings"]:
                self.samples.append((i, "findings"))
            if self.ds[i]["impression"]:
                self.samples.append((i, "impression"))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ds_idx, field = self.samples[idx]

        sample = self.ds[ds_idx]
        img = sample["image"]          # load tại đây
        text = sample[field]

        img = self.size_transform(img).convert("RGB")
        img = _toTensor(img)

        return {
            "image": img,
            "text": text,
            "idx": idx
        }



class ENTREPDataset(Dataset):
    def __init__(
        self,
        size_transform,
        img_dir="/kaggle/input/entrep-train-contrastive/Dataset/images",
        annotation_file="/kaggle/input/entrep-train-contrastive/Dataset/data.csv",
    ):
        self.img_dir = img_dir
        self.annotation_file = annotation_file
        self.size_transform = size_transform

        self.samples = []

        with open(self.annotation_file, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                path = row["path"].replace('image', "Image")
                caption = row["caption"]

                img_path = os.path.join(self.img_dir, path)
                self.samples.append((img_path, caption))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, text = self.samples[idx]

        img = self.size_transform(Image.open(img_path)).convert("RGB")
        img = _toTensor(img)

        return {
            "image": img,
            "text": text,
            "idx": idx
        }



In [ ]:
from tqdm import tqdm
import torch
import torch.nn.functional as F

def clip_infonce_loss(image_feats, text_feats, temperature=0.07):
    """
    image_feats: (B, D)
    text_feats:  (B, D)
    """

    # similarity matrix: (B, B)
    logits = image_feats @ text_feats.t() / temperature

    labels = torch.arange(logits.size(0), device=logits.device)

    loss_t2i = F.cross_entropy(logits.t(), labels) # z_q * Z_T
    loss_i2t = F.cross_entropy(logits, labels)

    return 0.5 * (loss_t2i + loss_i2t)


def train_denoiser(
    denoiser,
    model,
    dataloader,
    optimizer,
    device="cuda",
    epochs=10,
    epsilon=0.03,
    alpha=1.0,          # weight for InfoNCE
    beta=1.0,           # weight for MSE
    temperature=0.07,
):
    """
    alpha * InfoNCE + beta * MSE
    """

    denoiser.train()
    for epoch in range(epochs):
        total_loss = 0.0
        total_cl = 0.0
        total_mse = 0.0

        for batch in tqdm(dataloader):
            imgs = batch["image"].to(device)   # (B,C,H,W)

            # merge texts
            texts =  batch["text"]

            # ===== add noise =====
            noise = torch.randn_like(imgs)
            imgs_noisy = (imgs + epsilon * noise).clamp(0, 1)

            # ===== denoise =====
            imgs_denoised = denoiser(imgs_noisy)
            # ===== encode =====
            with torch.no_grad():
                text_feats = model.encode_text(texts)

            image_feats = model.encode_image(imgs_denoised)
            # ===== losses =====
            loss_cl = clip_infonce_loss(
                image_feats,
                text_feats,
                temperature=temperature
            )

            loss_mse = F.mse_loss(imgs_denoised, imgs)

            loss = alpha * loss_cl + beta * loss_mse
            # loss = loss_cl

            # ===== backward =====
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_cl += loss_cl.item()
            total_mse += loss_mse.item()

        print(
            f"[Epoch {epoch+1}/{epochs}] "
            f"Total: {total_loss/len(dataloader):.4f} | "
            f"CL: {total_cl/len(dataloader):.4f} | "
            f"MSE: {total_mse/len(dataloader):.4f}"
        )


In [ ]:
import torch
# config ..
epochs = 50
batch_size = 64
epsilon = 0.03
alpha = 0.5
model_name = 'medclip'
size_transform = SIZE_TRANSFORM[model_name]
dataset_name = 'mimic'
mode_pretrained = "ssl"

In [ ]:
from torch.utils.data import DataLoader
import torch
import yaml

if model_name in ['medclip', 'biomedclip']:
    model = ModelFactory.create_model(
        model_type=model_name,
        variant='base',
        pretrained=True,
        mode_pretrained=mode_pretrained
    )

elif model_name == 'entrep':
    config_path = "configs/entrep_contrastive.yaml"
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    model_config = config.get('model', {})
    model = ModelFactory.create_model(
        model_type="entrep",
        variant='base',
        checkpoint=None,
        pretrained=False,
        **{k: v for k, v in model_config.items() if k != 'model_type' and k != "pretrained" and k != "checkpoint"},
        mode_pretrained=mode_pretrained
    )   

if dataset_name == "mimic":
    dataset = MiMic(
        size_transform=size_transform,
    )
elif dataset_name == "entrep":
    dataset = ENTREPDataset(
        size_transform=size_transform
    )



denoiser = Denoiser().cuda()

In [ ]:
dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    drop_last=True
)


optimizer = torch.optim.AdamW(
    denoiser.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

train_denoiser(
    denoiser,
    model,
    dataloader,
    optimizer,
    device="cuda",
    epochs=epochs,
    epsilon=epsilon,
    alpha=alpha,          # weight for InfoNCE
    beta=alpha,           # weight for MSE
)

In [ ]:
torch.save(denoiser.state_dict(), f"denoiser_{model_name}.pth")